# Sweep 13a — Architecture: Aggregators, Fusion, Copy Anatomy, Graph Depth

**Question:** Which architectural choices matter beyond modality?

**~45 runs × full epochs ≈ 7–8 hours**

| Group | Topic | Configs | Seeds | Runs |
|-------|-------|---------|-------|------|
| A | Aggregator type | 3 (last / attention / attention_residual) | 3 | 9 |
| B | Fusion strategy | 2 (film / additive_no_film) | 3 | 6 |
| C1 | Copy: per_visit_copy | 2 (on / off) | 3 | 6 |
| C2 | Copy: ar_max_seq_len | 4 (10 / 20 / 35 / 50) | 3 | 12 |
| D | Drug graph depth | 4 (hgt_layers 0–3) | 3 | 12 |

**Backbones (each group uses the minimal context to isolate the variable):**
- **A**: naked + labs200, no notes, no copy  → clean aggregator signal
- **B**: notes + labs200, no copy           → fusion on mixed-modal input
- **C**: notes + labs200 + copy             → full modality, vary copy internals
- **D**: naked (no labs, no notes, no copy) → pure graph depth effect

All groups share the standard backbone: `hgt_ehr_feat`, `transformer` encoder, `count_reg_weight 0.008`.
Group D sweeps `hgt_layers` 0/1/2/3; all other groups fix `hgt_layers=1`.

In [ ]:
import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
!pip install -q torch_geometric
!pip install -q pyyaml pandas numpy scikit-learn

In [ ]:
import os, sys, glob

if os.path.exists("/kaggle"):
    print("Running on Kaggle")
    os.chdir("/kaggle/working")
    os.system("rm -rf ./data ./src")
    os.makedirs("data/processed", exist_ok=True)
    os.makedirs("data/embeddings", exist_ok=True)

    train_paths = glob.glob("/kaggle/input/**/train.py", recursive=True)
    if not train_paths:
        raise FileNotFoundError("train.py not found in /kaggle/input")
    src_dir = os.path.dirname(train_paths[0])
    os.system(f"cp -r {src_dir} /kaggle/working/src")
    sys.path.append("/kaggle/working/src")

    for find_file in ["cohort_mimic3.pkl", "records_final.pkl"]:
        found = glob.glob(f"/kaggle/input/**/{find_file}", recursive=True)
        if found:
            processed_dir = os.path.dirname(found[0])
            print(f"Found processed dir at: {processed_dir}")
            for root, dirs, files in os.walk(processed_dir):
                rel = os.path.relpath(root, processed_dir)
                target_dir = os.path.join("./data/processed", rel) if rel != "." else "./data/processed"
                os.makedirs(target_dir, exist_ok=True)
                for fname in files:
                    src = os.path.join(root, fname)
                    link = os.path.join(target_dir, fname)
                    if not os.path.exists(link):
                        try:
                            os.symlink(src, link)
                        except OSError:
                            os.system(f"cp '{src}' '{link}'")
            break

    emb_paths = glob.glob("/kaggle/input/**/code_embeddings.pt", recursive=True)
    if not emb_paths:
        raise FileNotFoundError("code_embeddings.pt not found")
    emb_dir = os.path.dirname(emb_paths[0])
    for fpath in glob.glob(f"{emb_dir}/*"):
        fname = os.path.basename(fpath)
        link = f"./data/embeddings/{fname}"
        if not os.path.exists(link):
            os.symlink(fpath, link)

    # Note embeddings — needed for Group B and C
    note_paths = glob.glob("/kaggle/input/**/note_embeddings_mimic3.pkl", recursive=True)
    if note_paths:
        note_src = note_paths[0]
        note_link = "./data/processed/note_embeddings_mimic3.pkl"
        if not os.path.exists(note_link):
            os.symlink(note_src, note_link)
        print(f"Note embeddings linked: {note_src}")
    else:
        print("WARNING: note_embeddings_mimic3.pkl not found — Groups B and C will fail")

    reg_path = "/kaggle/working/src/model/registry.py"
    if os.path.exists(reg_path):
        with open(reg_path, "r") as rf:
            content = rf.read()
        if "import inspect" not in content:
            old = "        return cls(**kwargs)"
            new = (
                "        import inspect\n"
                "        sig = inspect.signature(cls.__init__)\n"
                "        has_var_keyword = any(p.kind == inspect.Parameter.VAR_KEYWORD "
                "for p in sig.parameters.values())\n"
                "        if has_var_keyword:\n"
                "            filtered_kwargs = kwargs\n"
                "        else:\n"
                "            filtered_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}\n"
                "        return cls(**filtered_kwargs)"
            )
            if old in content:
                content = content.replace(old, new)
                with open(reg_path, "w") as wf:
                    wf.write(content)
                print("  registry.py patched")

print("Setup done:", os.getcwd())

In [ ]:
import yaml, subprocess, gc, json, numpy as np
from pathlib import Path

BASE_CONFIG = "src/config.yaml"

def make_config(output_path, model_overrides=None, preprocessing_overrides=None, smoke_epochs=None):
    with open(BASE_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    if model_overrides:
        for k, v in model_overrides.items():
            cfg["model"][k] = v
    if preprocessing_overrides:
        for k, v in preprocessing_overrides.items():
            cfg["preprocessing"][k] = v
    if smoke_epochs is not None:
        cfg["training"]["epochs"] = smoke_epochs
    with open(output_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    return str(output_path)

# ── Shared backbone ────────────────────────────────────────────────────────────
# hgt_layers=1 everywhere except Group D which sweeps it
BACKBONE_COMMON = {
    "graph_encoder_type": "hgt_ehr_feat",
    "hgt_layers": 1,
    "pos_weight_cap": 15.0,
}
BACKBONE_FLAGS = (
    "--device cuda "
    "--visit_level_training "
)

# Labs reference used by Groups A, B, C
LAB_FLAGS = "--lab_pkl data/processed/lab_vectors_200labs.pkl --num_labs 200"

SEEDS = [42, 123, 456]
reports_dir = Path("experiment_reports/sweep_13a_architecture")
results_log = []

def run_group(configs, group_tag):
    """Run all (name, cfg_path, flags) combos × SEEDS, log to results_log."""
    import torch
    for name, cfg_path, extra_flags in configs:
        for seed in SEEDS:
            run_name = f"{name}_seed{seed}"
            run_dir  = reports_dir / run_name
            run_dir.mkdir(parents=True, exist_ok=True)
            log_path = run_dir / "training_log.txt"
            cmd = (
                f"python -u src/train.py --config {cfg_path} "
                f"{BACKBONE_FLAGS} {extra_flags} "
                f"--seed {seed} --results_dir {run_dir}"
            )
            print(f"\n=== [{group_tag}] {run_name} ===\n>> {cmd}\n")
            try:
                with open(log_path, "w") as lf:
                    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                                            stderr=subprocess.STDOUT, text=True)
                    for line in proc.stdout:
                        print(line, end="")
                        lf.write(line)
                    proc.wait()
                status = "SUCCESS" if proc.returncode == 0 else f"FAILED ({proc.returncode})"
                results_log.append(f"{status}: {run_name}")
            except Exception as e:
                results_log.append(f"CRASH: {run_name}: {e}")
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

print(f"Config helpers ready. Reports → {reports_dir}")

In [ ]:
# ── Smoke test: 1 epoch for one rep from each group ───────────────────────────
Path("smoke_s13a").mkdir(exist_ok=True)

SMOKE_CASES = [
    # (label, model_overrides, prep_overrides, extra_flags)
    ("A_attn_res",
     {"copy_mechanism": False, **BACKBONE_COMMON},
     {"note_method": "none", "lab_dim": 400},
     f"--aggregator_type attention_residual --fusion_strategy film {LAB_FLAGS}"),

    ("B_film",
     {"copy_mechanism": False, **BACKBONE_COMMON},
     {"lab_dim": 400},
     f"--fusion_strategy film {LAB_FLAGS}"),

    ("C_copy_on",
     {"copy_mechanism": True, "per_visit_copy": True, **BACKBONE_COMMON},
     {"lab_dim": 400},
     f"--fusion_strategy film {LAB_FLAGS}"),

    ("D_depth1",
     {"copy_mechanism": False, "hgt_layers": 1, **{k: v for k, v in BACKBONE_COMMON.items() if k != 'hgt_layers'}},
     {"note_method": "none", "lab_dim": 0},
     "--fusion_strategy film"),
]

smoke_results = []
for label, m_ov, p_ov, flags in SMOKE_CASES:
    cfg_path = f"s13a_smoke_{label}.yaml"
    make_config(cfg_path, model_overrides=m_ov, preprocessing_overrides=p_ov, smoke_epochs=1)
    cmd = (f"python -u src/train.py --config {cfg_path} "
           f"{BACKBONE_FLAGS} {flags} "
           f"--seed 42 --results_dir smoke_s13a/{label}")
    print(f"SMOKE {label}: {cmd}")
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (proc.stdout + proc.stderr).strip().split("\n")[-5:]
    for l in tail: print(l)
    status = "PASS" if proc.returncode == 0 else f"FAIL ({proc.returncode})"
    smoke_results.append(f"{status}: {label}")
    print(f">>> {status}\n")

for r in smoke_results: print(r)
if any("FAIL" in r for r in smoke_results):
    raise RuntimeError("Smoke tests failed — fix before full run.")
print("All smoke tests passed.")

In [ ]:
# ── Group A: Aggregator type ───────────────────────────────────────────────────
# Backbone: naked + labs200, no notes, no copy → isolates aggregator impact
# Varies: --aggregator_type {last, attention, attention_residual}
# hgt_layers=1 fixed; fusion_strategy=film (only one modality so FiLM is trivial)

print("\n" + "#"*70)
print("# GROUP A — AGGREGATOR TYPE (9 runs)")
print("#"*70)

AGGREGATORS = [
    ("last",               "A_last"),
    ("attention",          "A_attn"),
    ("attention_residual", "A_attn_res"),
]

group_a_configs = []
for agg_name, short in AGGREGATORS:
    cfg_path = f"s13a_{short}.yaml"
    make_config(cfg_path,
                model_overrides={"copy_mechanism": False, **BACKBONE_COMMON},
                preprocessing_overrides={"note_method": "none", "lab_dim": 400})
    flags = f"--aggregator_type {agg_name} --fusion_strategy film {LAB_FLAGS}"
    group_a_configs.append((short, cfg_path, flags))
    print(f"  {short}: aggregator={agg_name}")

run_group(group_a_configs, "A")
print(f"\nGroup A done. Total so far: {len(results_log)} runs")

In [ ]:
# ── Group B: Fusion strategy ───────────────────────────────────────────────────
# Backbone: notes + labs200, no copy → FiLM vs additive gate on mixed-modal input
# The point: does FiLM conditioning (γ/β from code reprs) beat a simple additive gate?

print("\n" + "#"*70)
print("# GROUP B — FUSION STRATEGY (6 runs)")
print("#"*70)

FUSION_STRATEGIES = [
    ("film",           "B_film"),
    ("additive_no_film", "B_additive"),
]

group_b_configs = []
for fs_name, short in FUSION_STRATEGIES:
    cfg_path = f"s13a_{short}.yaml"
    make_config(cfg_path,
                model_overrides={"copy_mechanism": False, **BACKBONE_COMMON},
                preprocessing_overrides={"lab_dim": 400})  # note_method default = clinicalbert
    flags = f"--fusion_strategy {fs_name} {LAB_FLAGS}"
    group_b_configs.append((short, cfg_path, flags))
    print(f"  {short}: fusion_strategy={fs_name}")

run_group(group_b_configs, "B")
print(f"\nGroup B done. Total so far: {len(results_log)} runs")

In [ ]:
# ── Group C1: Copy anatomy — per_visit_copy ────────────────────────────────────
# Backbone: notes + labs200 + copy ON
# per_visit_copy=True  → copy queries built per-visit from patient repr
# per_visit_copy=False → single global copy query from final repr

print("\n" + "#"*70)
print("# GROUP C1 — COPY: per_visit_copy (6 runs)")
print("#"*70)

COPY_PVC = [
    (True,  "C1_pvc_on"),
    (False, "C1_pvc_off"),
]

group_c1_configs = []
for pvc_val, short in COPY_PVC:
    cfg_path = f"s13a_{short}.yaml"
    make_config(cfg_path,
                model_overrides={
                    "copy_mechanism": True,
                    "per_visit_copy": pvc_val,
                    **BACKBONE_COMMON
                },
                preprocessing_overrides={"lab_dim": 400})
    flags = f"--fusion_strategy film {LAB_FLAGS}"
    group_c1_configs.append((short, cfg_path, flags))
    print(f"  {short}: per_visit_copy={pvc_val}")

run_group(group_c1_configs, "C1")
print(f"\nGroup C1 done. Total so far: {len(results_log)} runs")

In [ ]:
# ── Group C2: Copy anatomy — ar_max_seq_len ────────────────────────────────────
# How long a look-back window does the autoregressive copy need?
# Default is 35. Test shorter (10, 20) and longer (50).
# Backbone same as C1: notes + labs200 + copy ON, per_visit_copy=True

print("\n" + "#"*70)
print("# GROUP C2 — COPY: ar_max_seq_len (12 runs)")
print("#"*70)

SEQ_LENS = [10, 20, 35, 50]

group_c2_configs = []
for seq_len in SEQ_LENS:
    short = f"C2_seq{seq_len}"
    cfg_path = f"s13a_{short}.yaml"
    make_config(cfg_path,
                model_overrides={
                    "copy_mechanism": True,
                    "per_visit_copy": True,
                    "ar_max_seq_len": seq_len,
                    **BACKBONE_COMMON
                },
                preprocessing_overrides={"lab_dim": 400})
    flags = f"--fusion_strategy film {LAB_FLAGS}"
    group_c2_configs.append((short, cfg_path, flags))
    print(f"  {short}: ar_max_seq_len={seq_len}")

run_group(group_c2_configs, "C2")
print(f"\nGroup C2 done. Total so far: {len(results_log)} runs")

In [ ]:
# ── Group D: Drug graph depth ──────────────────────────────────────────────────
# Backbone: naked (no labs, no notes, no copy) → purest signal on graph depth
# hgt_layers=0 means no HGT message passing — just initial Morgan embeddings
# hgt_layers=1 is the current default in all other sweeps
# hgt_layers=2 is the config.yaml default
# hgt_layers=3 tests over-smoothing risk

print("\n" + "#"*70)
print("# GROUP D — DRUG GRAPH DEPTH: hgt_layers 0/1/2/3 (12 runs)")
print("#"*70)

GRAPH_DEPTHS = [0, 1, 2, 3]

group_d_configs = []
for depth in GRAPH_DEPTHS:
    short = f"D_depth{depth}"
    cfg_path = f"s13a_{short}.yaml"
    make_config(cfg_path,
                model_overrides={
                    "copy_mechanism": False,
                    "graph_encoder_type": "hgt_ehr_feat",
                    "hgt_layers": depth,
                    "pos_weight_cap": 15.0,
                },
                preprocessing_overrides={"note_method": "none", "lab_dim": 0})
    flags = "--fusion_strategy film"
    group_d_configs.append((short, cfg_path, flags))
    print(f"  {short}: hgt_layers={depth}")

run_group(group_d_configs, "D")
print(f"\nGroup D done. Total so far: {len(results_log)} runs")

In [ ]:
# ── Results compiler ───────────────────────────────────────────────────────────
import json, numpy as np
from pathlib import Path

reports_dir = Path("experiment_reports/sweep_13a_architecture")

def get_metric(d, key):
    sec = d.get("test_metrics", {})
    for k in [key, key.replace("_", " "), key.title()]:
        if k in sec: return float(sec[k])
        if k in d:   return float(d[k])
    return 0.0

all_results = {}
for jp in sorted(reports_dir.rglob("result_*.json")):
    with open(jp) as f: d = json.load(f)
    run_name = jp.parent.name
    idx = run_name.rfind("_seed")
    if idx == -1: continue
    all_results.setdefault(run_name[:idx], []).append(d)

def summarize(name):
    runs = all_results.get(name, [])
    if not runs: return None
    jac  = [get_metric(d, "jaccard")  for d in runs]
    f1   = [get_metric(d, "f1")       for d in runs]
    ddi  = [get_metric(d, "ddi_rate") for d in runs]
    return {
        "jac_mean": float(np.mean(jac)),
        "jac_std":  float(np.std(jac, ddof=1) if len(jac) > 1 else 0.0),
        "f1":       float(np.mean(f1)),
        "ddi":      float(np.mean(ddi)),
        "n":        len(jac),
    }

def print_table(header, rows):
    """rows = [(label, config_key)] sorted by preference"""
    print(f"\n  {header}")
    print(f"  {'Config':<28} {'Jac mean±std':>16}  {'Δ best':>8}  F1      DDI     n")
    print(f"  {'-'*28} {'-'*16}  {'-'*8}  {'-'*6}  {'-'*6}  -")
    sums = [(label, key, summarize(key)) for label, key in rows]
    valid = [(l, k, s) for l, k, s in sums if s]
    if not valid:
        print("  (no results yet)")
        return None
    best_jac = max(s["jac_mean"] for _, _, s in valid)
    for label, key, s in sums:
        if s is None:
            print(f"  {label:<28} {'(missing)':>16}")
            continue
        delta = f"{s['jac_mean'] - best_jac:+.4f}"
        marker = " ←best" if s["jac_mean"] == best_jac else ""
        print(f"  {label:<28} {s['jac_mean']:.4f}±{s['jac_std']:.4f}  {delta:>8}  "
              f"{s['f1']:.4f}  {s['ddi']:.4f}  {s['n']}{marker}")
    return best_jac

print("\n" + "="*80)
print("SWEEP 13a — ARCHITECTURE RESULTS")
print("="*80)

# ── Group A ──
print_table(
    "GROUP A — Aggregator type  (backbone: naked+labs200, no copy)",
    [("A: last",               "A_last"),
     ("A: attention",          "A_attn"),
     ("A: attention_residual", "A_attn_res")]
)

# ── Group B ──
print_table(
    "GROUP B — Fusion strategy  (backbone: notes+labs200, no copy)",
    [("B: film",           "B_film"),
     ("B: additive_no_film", "B_additive")]
)

# ── Group C1 ──
print_table(
    "GROUP C1 — per_visit_copy  (backbone: notes+labs200+copy)",
    [("C1: per_visit_copy=True",  "C1_pvc_on"),
     ("C1: per_visit_copy=False", "C1_pvc_off")]
)

# ── Group C2 ──
print_table(
    "GROUP C2 — ar_max_seq_len  (backbone: notes+labs200+copy)",
    [(f"C2: seq_len={sl}", f"C2_seq{sl}") for sl in [10, 20, 35, 50]]
)

# ── Group D ──
print_table(
    "GROUP D — Graph depth hgt_layers  (backbone: naked only)",
    [(f"D: hgt_layers={d}", f"D_depth{d}") for d in [0, 1, 2, 3]]
)

# ── Summary line ──
print("\n" + "─"*80)
print("RECOMMENDED ARCHITECTURE CHOICES (fill in after results):")
print("  Aggregator  : ___  (Group A best)")
print("  Fusion      : ___  (Group B best)")
print("  per_visit_copy : ___  (Group C1 best)")
print("  ar_max_seq_len : ___  (Group C2 best)")
print("  hgt_layers  : ___  (Group D best)")
print("─"*80)

In [ ]:
# ── Run log summary ────────────────────────────────────────────────────────────
total    = len(results_log)
success  = sum(1 for r in results_log if "SUCCESS" in r)
failed   = sum(1 for r in results_log if "FAILED"  in r)
crashed  = sum(1 for r in results_log if "CRASH"   in r)

print(f"\nRun log: {total} total | {success} success | {failed} failed | {crashed} crashed")
if failed + crashed > 0:
    print("\nFailed/crashed runs:")
    for r in results_log:
        if "SUCCESS" not in r: print(f"  {r}")

In [ ]:
import zipfile
from pathlib import Path

zip_name = "reports_sweep_13a_architecture.zip"
rd = Path("experiment_reports/sweep_13a_architecture")
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(rd.rglob("result_*.json")):
        zf.write(p, p.relative_to(rd))
    for p in sorted(rd.rglob("training_log.txt")):
        zf.write(p, p.relative_to(rd))
print(f"Exported → {zip_name}")